# 🔍 Live Fynd-detektor — Aktiva annonser i Örebro

**Syfte:** Scrapa alla bostäder till salu just nu, köra ML-modellen på dem,  
och identifiera undervärderade bostäder (fynd!)  

**Output:** `active_listings_scored.csv` — redo för dashboarden  
**Kör dagligen** för att hålla datan uppdaterad

In [1]:
# ============================================================
# STEG 1: Importera paket
# ============================================================
import pandas as pd
import numpy as np
import time
import random
import re
import json
import os
import joblib
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

print('Paket laddade!')
print(f'Datum: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

Paket laddade!
Datum: 2026-03-17 14:39


In [2]:
# ============================================================
# STEG 2: Selenium-driver + scraper
# ============================================================
def create_driver():
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                         'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def scrape_active_listings(driver, url):
    """Scrapa en sida med aktiva annonser."""
    driver.get(url)
    time.sleep(random.uniform(2, 4))
    soup = BeautifulSoup(driver.page_source, 'lxml')
    cards = soup.find_all('a', href=re.compile(r'/bostad/'))
    listings = []
    for card in cards:
        text = card.get_text(' ', strip=True)
        if len(text) < 10:
            continue
        listing = {'raw_text': text, 'url': 'https://www.hemnet.se' + card.get('href', '')}
        price_match = re.search(r'([\d\s]+)\s*kr', text)
        if price_match:
            p = price_match.group(1).replace(' ', '').replace('\xa0', '')
            try:
                listing['utgangspris'] = int(p)
            except:
                pass
        area_match = re.search(r'([\d,\.]+)\s*m[²2]', text)
        if area_match:
            listing['boarea_kvm'] = float(area_match.group(1).replace(',', '.'))
        rooms_match = re.search(r'([\d,]+)\s*rum', text)
        if rooms_match:
            listing['antal_rum'] = float(rooms_match.group(1).replace(',', '.'))
        fee_match = re.search(r'([\d\s]+)\s*kr/m[åa]n', text)
        if fee_match:
            f = fee_match.group(1).replace(' ', '').replace('\xa0', '')
            try:
                listing['avgift_kr'] = int(f)
            except:
                pass
        if listing.get('utgangspris') and listing.get('utgangspris') > 100000:
            listings.append(listing)
    return listings

print('Driver och scraper redo!')

Driver och scraper redo!


In [3]:
# ============================================================
# STEG 3: Scrapa alla aktiva annonser
# ============================================================
URLS = {
    'lagenheter': 'https://www.hemnet.se/till-salu/lagenhet/orebro-kommun',
    'villor': 'https://www.hemnet.se/till-salu/villa/orebro-kommun',
    'radhus': 'https://www.hemnet.se/till-salu/radhus/orebro-kommun',
}

driver = create_driver()
all_active = []

try:
    for typ, base_url in URLS.items():
        print(f'Scrapar {typ}...')
        for page in range(1, 11):
            url = f'{base_url}?page={page}'
            listings = scrape_active_listings(driver, url)
            if not listings:
                print(f'  Sida {page}: inga fler resultat')
                break
            for l in listings:
                l['bostadstyp'] = typ
            all_active.extend(listings)
            print(f'  Sida {page}: {len(listings)} annonser')
finally:
    driver.quit()

df_active = pd.DataFrame(all_active)
print(f'\nTotalt: {len(df_active)} aktiva annonser')

Scrapar lagenheter...
  Sida 1: 55 annonser
  Sida 2: 54 annonser
  Sida 3: 54 annonser
  Sida 4: 54 annonser
  Sida 5: 52 annonser
  Sida 6: 52 annonser
  Sida 7: 53 annonser
  Sida 8: 53 annonser
  Sida 9: 54 annonser
  Sida 10: 54 annonser
Scrapar villor...
  Sida 1: 55 annonser
  Sida 2: 54 annonser
  Sida 3: 53 annonser
  Sida 4: 12 annonser
  Sida 5: 12 annonser
  Sida 6: 12 annonser
  Sida 7: 12 annonser
  Sida 8: 12 annonser
  Sida 9: 12 annonser
  Sida 10: 12 annonser
Scrapar radhus...
  Sida 1: 54 annonser
  Sida 2: 54 annonser
  Sida 3: 25 annonser
  Sida 4: 25 annonser
  Sida 5: 25 annonser
  Sida 6: 25 annonser
  Sida 7: 25 annonser
  Sida 8: 25 annonser
  Sida 9: 25 annonser
  Sida 10: 25 annonser

Totalt: 1089 aktiva annonser


In [4]:
# ============================================================
# STEG 4: Rensa och förbered data
# ============================================================

df_live = df_active.copy()

# Fyll saknade avgifter
median_avgift_lag = df_live.loc[df_live['bostadstyp'] == 'lagenheter', 'avgift_kr'].median()
median_avgift_rad = df_live.loc[df_live['bostadstyp'] == 'radhus', 'avgift_kr'].median()
df_live.loc[(df_live['bostadstyp'] == 'lagenheter') & df_live['avgift_kr'].isna(), 'avgift_kr'] = median_avgift_lag
df_live.loc[(df_live['bostadstyp'] == 'radhus') & df_live['avgift_kr'].isna(), 'avgift_kr'] = median_avgift_rad
df_live.loc[df_live['bostadstyp'] == 'villor', 'avgift_kr'] = \
    df_live.loc[df_live['bostadstyp'] == 'villor', 'avgift_kr'].fillna(0)

# Ta bort dubbletter och rader utan grunddata
df_live = df_live.drop_duplicates(subset='url')
df_live = df_live.dropna(subset=['utgangspris', 'boarea_kvm', 'antal_rum'])

# Extrahera område från URL
def extract_area_from_url(url):
    try:
        parts = url.split('/')[-1]
        cleaned = re.sub(r'^(lagenhet|villa|radhus)-\d+rum-', '', parts)
        area = cleaned.split('-orebro-kommun')[0]
        area = area.replace('-', ' ').title()
        return area
    except:
        return 'Örebro'

df_live['omrade_from_url'] = df_live['url'].apply(extract_area_from_url)

print(f'Rensade annonser: {len(df_live)}')
print(f'Bostadstyper: {df_live["bostadstyp"].value_counts().to_dict()}')
print(f'Saknade värden: {df_live[["utgangspris", "boarea_kvm", "antal_rum", "avgift_kr"]].isnull().sum().sum()}')

Rensade annonser: 717
Bostadstyper: {'lagenheter': 488, 'villor': 140, 'radhus': 89}
Saknade värden: 0


In [5]:
# ============================================================
# STEG 5: Smart områdesmatchning (URL → modellens omrade_grupp)
# ============================================================

url_to_area = {
    'sorbyangen': 'Sörbyängen',
    'sodra ladugardsangen': 'Ladugårdsängen',
    'ladugardsangen': 'Ladugårdsängen',
    'ormesta': 'Ormesta',
    'centralt vaster': 'Centralt Väster',
    'almby': 'Almby',
    'lillan': 'Lillån',
    'oster': 'Öster',
    'orebro': 'Örebro',
    'vaster': 'Väster',
    'adolfsberg': 'Adolfsberg',
    'rynningeasen': 'Rynningeåsen',
    'centralt oster': 'Centralt Öster',
    'mellringe': 'Mellringe',
    'solhaga': 'Solhaga',
    'bettorp': 'Bettorp',
    'sorby': 'Sörby',
    'centralt': 'Centralt',
    'sodra lindhult': 'Södra Lindhult',
    'ekeby almby': 'Almby',
    'hovsta': 'Hovsta',
    'lundby': 'Lundby',
    'bjorkhaga': 'Björkhaga',
    'nya hjarsta': 'Nya Hjärsta',
    'garphyttan': 'Garphyttan',
    'glanshammar': 'Glanshammar',
    'stora mellosa': 'Stora Mellösa',
    'vintrosa': 'Vintrosa',
    'odensbacken': 'Odensbacken',
    'mosas': 'Mosås',
    'marieberg': 'Marieberg',
    'rynninge': 'Rynninge',
    'vivalla': 'Vivalla',
    'brickebacken': 'Brickebacken',
    'baronbackarna': 'Baronbackarna',
    'hagaby': 'Hagaby',
    'norra bro': 'Norra Bro',
    'tybble': 'Tybble',
    'kilsmo': 'Kilsmo',
    'asker': 'Asker',
    'sodra ladugards': 'Ladugårdsängen',
    'ekeby': 'Almby',
    'norr': 'Norr',
    'soder': 'Söder',
    'talby': 'Talby',
    'klockhammar': 'Klockhammar',
    'nasta': 'Nästa',
    'latorp': 'Latorp',
    'askers by': 'Asker',
    'varberga': 'Varberga',
    'oxhagen': 'Oxhagen',
    'markbacken': 'Markbacken',
    'norrby': 'Norrby',
    'brickeberg': 'Brickeberg',
    'navesta': 'Navesta',
    'lillans': 'Lillån',
    'gamla hjarsta': 'Gamla Hjärsta',
    'rynninge asen': 'Rynningeåsen',
    'tegnerlunden': 'Tegnérlunden',
    'skogsangen': 'Skogsängen',
}

# Ladda modellens top_areas (70 st)
df_train = pd.read_csv('../data/processed/orebro_housing_enriched.csv')
top_areas = df_train['omrade_clean'].value_counts().head(70).index.tolist()

def match_area(url_area):
    clean = url_area.lower().strip()
    if clean in url_to_area:
        return url_to_area[clean]
    for key, value in url_to_area.items():
        if key in clean or clean in key:
            return value
    for area in top_areas:
        area_ascii = area.lower().replace('å', 'a').replace('ä', 'a').replace('ö', 'o').replace('é', 'e')
        if area_ascii in clean or clean in area_ascii:
            return area
    return 'övrigt'

df_live['omrade_matched'] = df_live['omrade_from_url'].apply(match_area)

matched = (df_live['omrade_matched'] != 'övrigt').sum()
print(f'Matchade: {matched} av {len(df_live)} ({matched/len(df_live)*100:.0f}%)')
print(f'övrigt: {(df_live["omrade_matched"] == "övrigt").sum()}')
print(f'\nTopp 10 matchade:')
print(df_live['omrade_matched'].value_counts().head(10))

Matchade: 653 av 717 (91%)
övrigt: 64

Topp 10 matchade:
omrade_matched
Örebro             73
övrigt             64
Ladugårdsängen     60
Sörbyängen         49
Ormesta            38
Almby              38
Öster              28
Centralt           27
Centralt Väster    26
Lillån             26
Name: count, dtype: int64


In [6]:
# ============================================================
# STEG 6: Ladda modell och bygg features
# ============================================================

model_data = joblib.load('../models/best_model.pkl')
model = model_data['model']
feature_names = model_data['feature_names']

print(f'Modell: {model_data.get("model_name", "unknown")}')
print(f'R²: {model_data["metrics"]["R2"]}')
print(f'Features: {len(feature_names)}')

df_pred = df_live.copy()

# Grundfeatures
df_pred['prisforandring_pct'] = 0
df_pred['sald_ar'] = 2026
df_pred['sald_manad'] = datetime.now().month

# Detaljfeatures — medianer per bostadstyp
df_pred['bostad_alder'] = df_train['bostad_alder'].median()
df_pred['har_hiss'] = (df_pred['bostadstyp'] == 'lagenheter').astype(int)
df_pred['har_balkong'] = (df_pred['bostadstyp'] == 'lagenheter').astype(int)
df_pred['har_garage'] = (df_pred['bostadstyp'] == 'villor').astype(int)
df_pred['renoverad'] = 0
df_pred['driftkostnad_ar'] = df_pred['bostadstyp'].map({
    'lagenheter': df_train.loc[df_train['bostadstyp'] == 'lagenheter', 'driftkostnad_ar'].median(),
    'villor': df_train.loc[df_train['bostadstyp'] == 'villor', 'driftkostnad_ar'].median(),
    'radhus': df_train.loc[df_train['bostadstyp'] == 'radhus', 'driftkostnad_ar'].median(),
})
df_pred['tomtarea_kvm'] = df_pred['bostadstyp'].map({
    'lagenheter': 0,
    'villor': df_train.loc[df_train['bostadstyp'] == 'villor', 'tomtarea_kvm'].median(),
    'radhus': df_train.loc[df_train['bostadstyp'] == 'radhus', 'tomtarea_kvm'].median(),
})
df_pred['vaning'] = df_pred['bostadstyp'].apply(lambda x: 2 if x == 'lagenheter' else 0)
df_pred['antal_besok'] = df_train['antal_besok'].median()

# One-hot bostadstyp
df_pred['bostadstyp_radhus'] = (df_pred['bostadstyp'] == 'radhus').astype(int)
df_pred['bostadstyp_villor'] = (df_pred['bostadstyp'] == 'villor').astype(int)

# One-hot omrade_grupp — med korrekta matchade områden
for col in feature_names:
    if col.startswith('omrade_grupp_'):
        area_name = col.replace('omrade_grupp_', '')
        df_pred[col] = (df_pred['omrade_matched'] == area_name).astype(int)

# Fyll saknade features med 0
for col in feature_names:
    if col not in df_pred.columns:
        df_pred[col] = 0
df_pred[feature_names] = df_pred[feature_names].fillna(0)

X_live = df_pred[feature_names]
print(f'\nFeature-matris: {X_live.shape}')
print(f'Saknade värden: {X_live.isnull().sum().sum()}')

Modell: XGBoost (tuned)
R²: 0.748
Features: 48

Feature-matris: (717, 48)
Saknade värden: 0


In [7]:
# ============================================================
# STEG 7: Kör modellen och bedöm med rimlighetscheck
# ============================================================

df_live['estimerat_varde'] = model.predict(X_live)
df_live['skillnad_kr'] = df_live['estimerat_varde'] - df_live['utgangspris']
df_live['skillnad_pct'] = (
    (df_live['estimerat_varde'] - df_live['utgangspris']) / df_live['estimerat_varde'] * 100
).round(1)

def smart_bedomning(row):
    pct = row['skillnad_pct']
    if abs(pct) > 60:
        return '⚠️ Osäkert'
    elif pct > 15:
        return '🟢 Potentiellt fynd'
    elif pct > -5:
        return '🟡 Rimligt pris'
    else:
        return '🔴 Överprissatt'

df_live['bedomning'] = df_live.apply(smart_bedomning, axis=1)
df_live['scrape_datum'] = datetime.now().strftime('%Y-%m-%d')
df_live['omrade'] = df_live['omrade_matched']

print('=== RESULTAT ===')
print(f'Totalt: {len(df_live)} annonser\n')
print(df_live['bedomning'].value_counts())
print(f'\nMedian estimat:  {df_live["estimerat_varde"].median():,.0f} kr')
print(f'Median utgångs:  {df_live["utgangspris"].median():,.0f} kr')
print(f'Median skillnad: {df_live["skillnad_kr"].median():,.0f} kr')

=== RESULTAT ===
Totalt: 717 annonser

bedomning
🔴 Överprissatt        386
🟡 Rimligt pris        164
⚠️ Osäkert             89
🟢 Potentiellt fynd     78
Name: count, dtype: int64

Median estimat:  1,652,167 kr
Median utgångs:  1,950,000 kr
Median skillnad: -248,404 kr


In [8]:
# ============================================================
# STEG 8: Visa resultat
# ============================================================

show_cols = ['omrade', 'bostadstyp', 'utgangspris', 'estimerat_varde', 
             'skillnad_pct', 'boarea_kvm', 'antal_rum', 'bedomning', 'url']

fynd = df_live[df_live['bedomning'] == '🟢 Potentiellt fynd'].sort_values('skillnad_pct', ascending=False)
print(f'🟢 ALLA FYND: {len(fynd)} st')
print('=' * 80)
display(fynd[show_cols])

print(f'\n🟡 RIMLIGT PRISSATTA: {len(df_live[df_live["bedomning"] == "🟡 Rimligt pris"])} st')
print(f'🔴 ÖVERPRISSATTA: {len(df_live[df_live["bedomning"] == "🔴 Överprissatt"])} st')
print(f'⚠️ OSÄKERT: {len(df_live[df_live["bedomning"] == "⚠️ Osäkert"])} st')

# Vivalla-check
vivalla = df_live[df_live['omrade_matched'].str.contains('Vivalla', case=False, na=False)]
if len(vivalla) > 0:
    print(f'\n--- Vivalla-check ---')
    print(vivalla[['omrade', 'utgangspris', 'estimerat_varde', 'skillnad_pct', 'bedomning']].to_string())

🟢 ALLA FYND: 78 st


,omrade,bostadstyp,utgangspris,estimerat_varde,skillnad_pct,boarea_kvm,antal_rum,bedomning,url
598,Klockhammar,villor,1195000,2.945322e+06,59.4,85.0,3.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/villa-3rum-klockh...
617,Asker,villor,600000,1.479596e+06,59.4,5.0,4.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/villa-4rum-asker-...
461,Centralt,lagenheter,2195000,5.088106e+06,56.9,141.0,5.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/lagenhet-5rum-cen...
558,Asker,villor,1150000,2.644536e+06,56.5,70.0,5.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/villa-5rum-asker-...
693,övrigt,villor,1850000,4.192396e+06,55.9,130.0,4.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/villa-4rum-nora-p...
...,...,...,...,...,...,...,...,...,...
337,Sörbyängen,lagenheter,1750000,2.101884e+06,16.7,93.0,4.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/lagenhet-4rum-sor...
243,Lillån,lagenheter,1495000,1.785560e+06,16.3,78.0,3.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/lagenhet-3rum-lil...
307,Örebro,lagenheter,1695000,2.019217e+06,16.1,85.5,4.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/lagenhet-4rum-ore...
350,Örebro,lagenheter,1695000,2.016566e+06,15.9,111.0,4.0,🟢 Potentiellt fynd,https://www.hemnet.se/bostad/lagenhet-4rum-ore...



🟡 RIMLIGT PRISSATTA: 164 st
🔴 ÖVERPRISSATTA: 386 st
⚠️ OSÄKERT: 89 st

--- Vivalla-check ---
      omrade  utgangspris  estimerat_varde  skillnad_pct   bedomning
492  Vivalla       595000        2240304.0          73.4  ⚠️ Osäkert
499  Vivalla       490000        1404991.0          65.1  ⚠️ Osäkert


In [9]:
print('Vivalla i modellens features?')
print([f for f in feature_names if 'ivalla' in f.lower()])

print(f'\nVivalla i träningsdata:')
print(df_train['omrade_clean'].value_counts().get('Vivalla', 0), 'bostäder')

Vivalla i modellens features?
[]

Vivalla i träningsdata:
2 bostäder


In [11]:
# ============================================================
# STEG 9: Spara resultaten
# ============================================================

save_cols = ['url', 'omrade', 'bostadstyp', 'utgangspris', 'estimerat_varde',
             'skillnad_kr', 'skillnad_pct', 'bedomning', 'boarea_kvm', 
             'antal_rum', 'avgift_kr', 'scrape_datum']

output_path = '../data/processed/active_listings_scored.csv'
df_live[save_cols].to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'Sparad: {output_path}')
print(f'Annonser: {len(df_live)}')
print(f'Datum: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'\nFördelning:')
print(df_live['bedomning'].value_counts().to_string())

print(f'\n{"="*50}')
print(f'KLART!')
print(f'Filen används av dashboarden.')
print(f'Kör denna notebook dagligen för uppdaterad data.')
print(f'{"="*50}')

Sparad: ../data/processed/active_listings_scored.csv
Annonser: 717
Datum: 2026-03-17 14:46

Fördelning:
bedomning
🔴 Överprissatt        386
🟡 Rimligt pris        164
⚠️ Osäkert             89
🟢 Potentiellt fynd     78

KLART!
Filen används av dashboarden.
Kör denna notebook dagligen för uppdaterad data.
